# VNP46A2 Stray-Light Diagnostic Comparison

This notebook renders the outputs from `stray_light_blackmarbler_redownload_diagnostic.R`. It is review-only: all geospatial processing and QA masking are done by the R diagnostic script.

Run first from the Reliability-Assessment repository root:

```bash
Rscript nightlight_downloader/Others/stray_light_blackmarbler_redownload_diagnostic.R
```

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PANEL_DATES = ["2023-10-20", "2023-02-28", "2023-01-29"]
CHECK_DATES = ["2023-10-20", "2023-02-28", "2023-04-28", "2023-11-06", "2023-01-27", "2023-01-29", "2023-10-22"]

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "STRAY_LIGHT_FIX_PLAN.md").exists() and (path / "blackmarbler").exists():
            return path
        nested = path / "6-codebases" / "repos" / "Reliability-Assessment"
        if (nested / "STRAY_LIGHT_FIX_PLAN.md").exists() and (nested / "blackmarbler").exists():
            return nested
    raise FileNotFoundError("Could not locate the Reliability-Assessment repository root.")

ROOT = find_repo_root(Path.cwd())
OUT_DIR = ROOT / "blackmarbler" / "out_vnp46a2_sa_daily" / "qa_straylight_validation"
PNG_DIR = OUT_DIR / "png"
SUMMARY_CSV = OUT_DIR / "diagnostic_summary.csv"
BIT_CSV = OUT_DIR / "diagnostic_bit_shares.csv"

print(f"Repository root: {ROOT}")
print(f"Diagnostic output: {OUT_DIR}")

In [ ]:
required = [SUMMARY_CSV, BIT_CSV] + [PNG_DIR / f"diagnostic_comparison_{date}.png" for date in PANEL_DATES]
missing = [path for path in required if not path.exists()]
if missing:
    message = "Missing diagnostic outputs. Run the R diagnostic script first.\n" + "\n".join(str(p) for p in missing)
    raise FileNotFoundError(message)

summary = pd.read_csv(SUMMARY_CSV)
bits = pd.read_csv(BIT_CSV)

summary["date"] = summary["date"].astype(str)
bits["date"] = bits["date"].astype(str)

print(f"Summary rows: {len(summary):,}")
print(f"Bit-share rows: {len(bits):,}")

## Side-by-Side Review Panels

Each row shows four panels for one date: current production radiance, fresh blackmarbler QF0-only radiance, fresh candidate Collection 2 QA-mask radiance, and the current-minus-candidate radiance difference. The R script fixes the radiance color scale across the three radiance panels for each date.

In [ ]:
table_cols = [
    "date", "scenario", "scope", "valid_share", "p_lit", "lit_share_all",
    "n_valid_pixels", "n_lit_pixels"
]

for date in PANEL_DATES:
    image_path = PNG_DIR / f"diagnostic_comparison_{date}.png"
    image = plt.imread(image_path)
    fig, ax = plt.subplots(figsize=(14, 9))
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(date)
    plt.show()

    rows = summary[(summary["date"] == date) & (summary["scope"].isin(["whole_scene", "rural_northern_cape_bbox"]))]
    rows = rows.sort_values(["scope", "scenario"])
    print(rows[table_cols].to_string(index=False))
    print()

## QA Bit Shares

Bit 13 is reported as `lunar_eclipse`. It is not labeled as stray light in this diagnostic.

In [ ]:
metrics = [
    "mandatory_qf", "snow_flag", "day_night", "cloud_confidence",
    "shadow", "cirrus", "snow_ice_cloud", "aurora", "lunar_eclipse"
]

bit_review = bits[(bits["date"].isin(PANEL_DATES)) & (bits["metric"].isin(metrics))].copy()
bit_review = bit_review.sort_values(["date", "metric", "value"])
bit_review[["date", "metric", "value", "label", "share_pixels", "bit_position", "keep_criterion"]]

## Quantitative Gates

These checks mirror the handoff plan. A failure here means the full production NTL pipeline should not be changed yet.

In [ ]:
def scenario_row(date: str, scenario: str, scope: str = "whole_scene") -> pd.Series:
    rows = summary[(summary["date"] == date) & (summary["scenario"] == scenario) & (summary["scope"] == scope)]
    if rows.empty:
        raise KeyError(f"Missing row for date={date}, scenario={scenario}, scope={scope}")
    return rows.iloc[0]

checks = []

row = scenario_row("2023-10-20", "blackmarbler_candidate_c2_mask", "whole_scene")
checks.append({"check": "2023-10-20 whole-scene p_lit", "value": row["p_lit"], "threshold": "< 0.08", "pass": row["p_lit"] < 0.08})
checks.append({"check": "2023-10-20 whole-scene lit_share_all", "value": row["lit_share_all"], "threshold": "< 0.05", "pass": row["lit_share_all"] < 0.05})

row = scenario_row("2023-01-27", "blackmarbler_candidate_c2_mask", "rural_northern_cape_bbox")
checks.append({"check": "2023-01-27 rural Northern Cape p_lit", "value": row["p_lit"], "threshold": "< 0.05", "pass": row["p_lit"] < 0.05})

for date in ["2023-10-20", "2023-02-28", "2023-04-28", "2023-11-06"]:
    row = scenario_row(date, "blackmarbler_candidate_c2_mask", "whole_scene")
    checks.append({"check": f"{date} candidate valid_share reported", "value": row["valid_share"], "threshold": "finite", "pass": pd.notna(row["valid_share"])})

pd.DataFrame(checks)